# Notebook 09 — Final Modelling Panel (Version 2)

Construct the Version 2 panel at reference quarter-end $t$ using the exact
timing design $B_{t-1}, M_{t-1} \rightarrow Y(t,t+365]$. Bank reports are
joined on `IDRSSD` and the exact preceding calendar quarter-end; a row shift
is never used. Macroeconomic variables arrive from Notebook 08 already
lagged once and are not lagged again.

This notebook preserves predictor missingness, performs a complete
source-versus-reference certificate scan, and enforces a one-time manual
adjudication gate before the modelling panel can be written. Failure dates,
source dates and availability margins remain in separate audit outputs.

In [ ]:
# Lock paths, calendars, feature manifests and audit contracts

from pathlib import Path
import importlib.util
import os

import numpy as np
import pandas as pd
from IPython.display import display


try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass


PROJECT_ROOT = Path(
    os.environ.get(
        "DISSERTATION_PROJECT_ROOT",
        os.environ.get("DATA_FINAL_ROOT", "/content/drive/MyDrive/Data Final"),
    )
).expanduser().resolve()

explicit_config = os.environ.get("V2_CONFIG_PATH")
config_candidates = [
    Path(explicit_config).expanduser() if explicit_config else None,
    PROJECT_ROOT / "config_v2.py",
    PROJECT_ROOT / "config.py",
    Path.cwd() / "config_v2.py",
    Path.cwd() / "config.py",
]
config_matches = []
for candidate in config_candidates:
    if candidate is not None and candidate.is_file():
        resolved = candidate.resolve()
        if resolved not in config_matches:
            config_matches.append(resolved)
if not config_matches:
    raise FileNotFoundError(
        "No shared configuration was found. Put config_v2.py under the "
        "project root, or set V2_CONFIG_PATH explicitly."
    )
CONFIG_PATH = config_matches[0]

config_spec = importlib.util.spec_from_file_location("dissertation_config_v2", CONFIG_PATH)
config_v2 = importlib.util.module_from_spec(config_spec)
config_spec.loader.exec_module(config_v2)

ANALYSIS_VERSION = getattr(config_v2, "ANALYSIS_VERSION", "v2_bank_lag1_main")
REFERENCE_CALENDAR_START = pd.Timestamp(
    getattr(config_v2, "REFERENCE_CALENDAR_START", "2001-06-30")
)
DEVELOPMENT_END = pd.Timestamp(getattr(config_v2, "DEVELOPMENT_END", "2018-12-31"))
FINAL_PURGE_START = pd.Timestamp(getattr(config_v2, "FINAL_PURGE_START", "2019-03-31"))
FINAL_PURGE_END = pd.Timestamp(getattr(config_v2, "FINAL_PURGE_END", "2019-12-31"))
FINAL_TEST_START = pd.Timestamp(getattr(config_v2, "FINAL_TEST_START", "2020-03-31"))
FINAL_TEST_END = pd.Timestamp(getattr(config_v2, "FINAL_TEST_END", "2025-06-30"))

NB05_OUTPUT_DIR = PROJECT_ROOT / "Notebook Outputs" / "05_Quarterly_Flows_and_Candidate_Predictors"
VERSION_OUTPUT_ROOT = PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION
NB07_OUTPUT_DIR = VERSION_OUTPUT_ROOT / "07_One_Year_Failure_Target"
NB08_OUTPUT_DIR = VERSION_OUTPUT_ROOT / "08_Macroeconomic_Data_Preparation"
NB09_OUTPUT_DIR = VERSION_OUTPUT_ROOT / "09_Final_Modelling_Panel"
NB09_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTOR_SOURCE_PATH = NB05_OUTPUT_DIR / "ffiec_commercial_bank_predictor_panel_2001Q1_2026Q1.csv"
FULL_WORKING_SOURCE_PATH = NB05_OUTPUT_DIR / "ffiec_commercial_bank_full_working_panel_2001Q1_2026Q1.parquet"
TARGET_PATH = NB07_OUTPUT_DIR / "bank_failure_target_1yr_2001Q1_2025Q2.parquet"
TARGET_AUDIT_PATH = NB07_OUTPUT_DIR / "bank_failure_target_1yr_audit_2001Q1_2025Q2.parquet"
MACRO_CANDIDATE_PATH = NB08_OUTPUT_DIR / "macro_predictors_lagged_candidate_reference.parquet"
MACRO_FULL_PATH = NB08_OUTPUT_DIR / "macro_predictors_lagged_full_observed_range.parquet"
MACRO_LAG_AUDIT_PATH = NB08_OUTPUT_DIR / "macro_predictors_lag_mapping_audit.csv"
MACRO_TIMING_AUDIT_PATH = NB08_OUTPUT_DIR / "cell_06_nominal_macro_publication_timing_audit.csv"

required_paths = [
    CONFIG_PATH, PREDICTOR_SOURCE_PATH, FULL_WORKING_SOURCE_PATH,
    TARGET_PATH, TARGET_AUDIT_PATH, MACRO_CANDIDATE_PATH,
    MACRO_FULL_PATH, MACRO_LAG_AUDIT_PATH, MACRO_TIMING_AUDIT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Notebook 09 prerequisites were not found:\n" + "\n".join(missing_paths))

BANK_PREDICTORS_V1 = [
    "log_total_assets", "equity_to_assets", "noncurrent_loans_to_loans",
    "allowance_to_loans", "provisions_to_average_assets",
    "other_real_estate_owned_to_assets", "return_on_average_assets",
    "net_interest_margin", "interest_expense_to_average_assets",
    "noninterest_income_to_average_assets", "efficiency_ratio",
    "loans_to_assets", "loans_to_deposits", "deposits_to_assets",
    "securities_to_assets", "aoci_to_assets", "annual_asset_growth",
    "annual_loan_growth", "annual_deposit_growth",
]
HISTORY_DEPENDENT_V1 = [
    "provisions_to_average_assets", "return_on_average_assets",
    "net_interest_margin", "interest_expense_to_average_assets",
    "noninterest_income_to_average_assets", "efficiency_ratio",
    "annual_asset_growth", "annual_loan_growth", "annual_deposit_growth",
]
MACRO_PREDICTORS = [
    "real_gdp_growth_lag1", "unemployment_rate_lag1",
    "effective_federal_funds_rate_lag1", "house_price_growth_lag1",
]
BANK_FEATURES_LAG1 = [f"{name}_lag1" for name in BANK_PREDICTORS_V1]
HISTORY_FEATURE_LAG1 = "insufficient_history_5q_lag1"
LOCKED_V1_NOTEBOOK09_CANDIDATE_MANIFEST = (
    BANK_PREDICTORS_V1 + MACRO_PREDICTORS + ["insufficient_history_5q"]
)
EXPECTED_V2_CANDIDATE_MANIFEST = (
    BANK_FEATURES_LAG1 + MACRO_PREDICTORS + [HISTORY_FEATURE_LAG1]
)
MODEL_FEATURES = EXPECTED_V2_CANDIDATE_MANIFEST.copy()
TARGET_DERIVED_FIELDS = {
    "failed_1yr", "fail_date", "failure_date", "days_to_failure",
    "target_reason", "target_window_end", "positive_horizon_days",
    "predictor_to_failure_days", "margin_after_30_day_due_date",
    "margin_after_35_day_due_date",
}
FEATURE_FORBIDDEN_FIELDS = {
    "idrssd", "cert", "CERT", "cert_source", "cert_reference",
    "report_date", "source_report_date", "expected_bank_source_date",
    "failure_date", "fail_date",
} | TARGET_DERIVED_FIELDS

PERIOD_WINDOWS = [
    ("fold1_train", "training", pd.Timestamp("2001-06-30"), pd.Timestamp("2008-12-31")),
    ("fold1_purge", "purge", pd.Timestamp("2009-03-31"), pd.Timestamp("2009-12-31")),
    ("fold1_validation", "validation", pd.Timestamp("2010-03-31"), pd.Timestamp("2012-12-31")),
    ("fold2_train", "training", pd.Timestamp("2001-06-30"), pd.Timestamp("2011-12-31")),
    ("fold2_purge", "purge", pd.Timestamp("2012-03-31"), pd.Timestamp("2012-12-31")),
    ("fold2_validation", "validation", pd.Timestamp("2013-03-31"), pd.Timestamp("2015-12-31")),
    ("fold3_train", "training", pd.Timestamp("2001-06-30"), pd.Timestamp("2014-12-31")),
    ("fold3_purge", "purge", pd.Timestamp("2015-03-31"), pd.Timestamp("2015-12-31")),
    ("fold3_validation", "validation", pd.Timestamp("2016-03-31"), DEVELOPMENT_END),
    ("final_train", "training", pd.Timestamp("2001-06-30"), DEVELOPMENT_END),
    ("final_purge", "purge", FINAL_PURGE_START, FINAL_PURGE_END),
    ("final_test", "test", FINAL_TEST_START, FINAL_TEST_END),
]

scope_register = pd.DataFrame({
    "control": [
        "Analysis timing", "Reference anchor", "Bank match key",
        "Bank-source date", "Bank lag applications", "Macro match key",
        "Macro lag applications", "Target timing", "Missing-value treatment",
        "Certificate mismatch policy", "Feature selection boundary",
    ],
    "locked_rule": [
        "B(t-1), M(t-1) -> Y(t, t+365 days]", "report_date t", "IDRSSD",
        "Exact preceding calendar quarter-end", 1, "reference report_date",
        1, "Target remains anchored at t", "Preserve; no imputation in Notebook 09",
        "Full scan; adjudicate to allowlist or exclusion list before panel write",
        "Carry the locked Version 1 Notebook 09 candidate manifest in lag1 form; final Version 2 selection is deferred to Notebook 11",
    ],
})

cell_01_checks = {
    "The analysis version is the V2 lagged-bank design": ANALYSIS_VERSION == "v2_bank_lag1_main",
    "The reference calendar begins in 2001Q2": REFERENCE_CALENDAR_START == pd.Timestamp("2001-06-30"),
    "All locked prerequisite handoffs exist": all(path.exists() for path in required_paths),
    "The locked Version 1 candidate manifest contains 24 features": len(LOCKED_V1_NOTEBOOK09_CANDIDATE_MANIFEST) == 24,
    "All 19 bank predictors are rendered in lag1 form": len(BANK_FEATURES_LAG1) == 19 and all(name.endswith("_lag1") for name in BANK_FEATURES_LAG1),
    "The history indicator is rendered in lag1 form": HISTORY_FEATURE_LAG1.endswith("_lag1"),
    "The four macro predictors arrive already lagged": len(MACRO_PREDICTORS) == 4 and all(name.endswith("_lag1") for name in MACRO_PREDICTORS),
    "No identifier, date or target-derived field is in the feature manifest": FEATURE_FORBIDDEN_FIELDS.isdisjoint(MODEL_FEATURES),
}
notebook_09_cell_01_gate = pd.DataFrame({
    "check": list(cell_01_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_01_checks.values()],
    "passed": list(cell_01_checks.values()),
})
scope_register.to_csv(NB09_OUTPUT_DIR / "cell_01_scope_register.csv", index=False)
notebook_09_cell_01_gate.to_csv(NB09_OUTPUT_DIR / "cell_01_setup_gate.csv", index=False)

print("Notebook 09 Cell 1 validation gate:")
display(notebook_09_cell_01_gate)
print("\nLocked timing and feature contract:")
with pd.option_context("display.max_colwidth", None):
    display(scope_register)
failed = [name for name, passed in cell_01_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 09 Cell 1 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 1 completed successfully.")
print("Output directory:", NB09_OUTPUT_DIR)

In [ ]:
# Load and validate the locked Notebook 05, 07 and 08 handoffs

if not notebook_09_cell_01_gate["passed"].all():
    raise AssertionError("Cell 2 blocked because Cell 1 did not pass.")


def normalise_quarter_end(frame, column="report_date"):
    result = frame.copy()
    result[column] = pd.to_datetime(result[column], errors="raise").dt.normalize()
    if not result[column].dt.is_quarter_end.all():
        raise AssertionError(f"{column} contains a non-quarter-end date.")
    return result


predictor_source = normalise_quarter_end(pd.read_csv(PREDICTOR_SOURCE_PATH, low_memory=False))
target_source = normalise_quarter_end(pd.read_parquet(TARGET_PATH))
target_audit_source = normalise_quarter_end(pd.read_parquet(TARGET_AUDIT_PATH))
macro_candidate_source = normalise_quarter_end(pd.read_parquet(MACRO_CANDIDATE_PATH))
macro_full_source = normalise_quarter_end(pd.read_parquet(MACRO_FULL_PATH))
macro_lag_source = normalise_quarter_end(
    pd.read_csv(MACRO_LAG_AUDIT_PATH, parse_dates=["report_date", "source_quarter"])
)
macro_timing_source = pd.read_csv(MACRO_TIMING_AUDIT_PATH)

for frame in [predictor_source, target_source, target_audit_source]:
    for column in ["cert", "idrssd"]:
        frame[column] = pd.to_numeric(frame[column], errors="raise").astype("int64")
target_source["failed_1yr"] = pd.to_numeric(target_source["failed_1yr"], errors="raise").astype("int8")
target_audit_source["failed_1yr"] = pd.to_numeric(target_audit_source["failed_1yr"], errors="raise").astype("int8")
target_audit_source["fail_date"] = pd.to_datetime(target_audit_source["fail_date"], errors="coerce").dt.normalize()
for column in BANK_PREDICTORS_V1:
    predictor_source[column] = pd.to_numeric(predictor_source[column], errors="coerce")
for frame in [macro_candidate_source, macro_full_source]:
    for column in MACRO_PREDICTORS:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

target_reference = target_source.loc[
    target_source["report_date"].ge(REFERENCE_CALENDAR_START)
].copy()
target_reference = target_reference.rename(columns={"cert": "cert_reference"})
target_reference = target_reference.sort_values(
    ["report_date", "cert_reference", "idrssd"], kind="mergesort"
).reset_index(drop=True)
target_reference["_reference_row_id"] = np.arange(len(target_reference), dtype="int64")

target_audit_reference = target_audit_source.loc[
    target_audit_source["report_date"].ge(REFERENCE_CALENDAR_START)
].rename(columns={"cert": "cert_reference"})

source_summary = pd.DataFrame({
    "source": ["Notebook 05 bank predictors", "Notebook 07 target", "Notebook 07 target audit", "Notebook 08 candidate macro", "Notebook 08 full macro"],
    "observations": [len(predictor_source), len(target_source), len(target_audit_source), len(macro_candidate_source), len(macro_full_source)],
    "first_date": [predictor_source.report_date.min(), target_source.report_date.min(), target_audit_source.report_date.min(), macro_candidate_source.report_date.min(), macro_full_source.report_date.min()],
    "last_date": [predictor_source.report_date.max(), target_source.report_date.max(), target_audit_source.report_date.max(), macro_candidate_source.report_date.max(), macro_full_source.report_date.max()],
})

cell_02_checks = {
    "Notebook 05 contains the three identifiers and all 19 bank predictors": set(["report_date", "cert", "idrssd"] + BANK_PREDICTORS_V1).issubset(predictor_source.columns),
    "Notebook 05 source IDRSSD-quarter keys are unique": not predictor_source.duplicated(["idrssd", "report_date"]).any(),
    "Notebook 07 primary keys are unique": not target_source.duplicated(["idrssd", "report_date"]).any(),
    "Notebook 07 audit keys are unique": not target_audit_source.duplicated(["idrssd", "report_date"]).any(),
    "Notebook 07 primary and audit rows agree": target_source[["report_date", "cert", "idrssd", "failed_1yr"]].sort_values(["report_date", "cert", "idrssd"]).reset_index(drop=True).equals(target_audit_source[["report_date", "cert", "idrssd", "failed_1yr"]].sort_values(["report_date", "cert", "idrssd"]).reset_index(drop=True)),
    "Exactly one Notebook 07 date is excluded by the V2 start restriction": target_source.loc[target_source.report_date.lt(REFERENCE_CALENDAR_START), "report_date"].nunique() == 1,
    "The excluded reference date is 2001Q1": set(target_source.loc[target_source.report_date.lt(REFERENCE_CALENDAR_START), "report_date"]) == {pd.Timestamp("2001-03-31")},
    "The restricted target begins in 2001Q2": target_reference.report_date.min() == REFERENCE_CALENDAR_START,
    "The restricted target ends at the configured test end": target_reference.report_date.max() == FINAL_TEST_END,
    "Both Notebook 08 macro handoffs have unique dates": macro_candidate_source.report_date.is_unique and macro_full_source.report_date.is_unique,
    "The candidate macro handoff contains the four already-lagged predictors": list(macro_candidate_source.columns) == ["report_date"] + MACRO_PREDICTORS,
    "No source bank predictor is infinite": not np.isinf(predictor_source[BANK_PREDICTORS_V1].to_numpy(float)).any(),
    "No macro value is missing or infinite": macro_candidate_source[MACRO_PREDICTORS].notna().all().all() and np.isfinite(macro_candidate_source[MACRO_PREDICTORS].to_numpy(float)).all(),
}
notebook_09_cell_02_gate = pd.DataFrame({
    "check": list(cell_02_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_02_checks.values()],
    "passed": list(cell_02_checks.values()),
})
source_summary.to_csv(NB09_OUTPUT_DIR / "cell_02_source_summary.csv", index=False)
notebook_09_cell_02_gate.to_csv(NB09_OUTPUT_DIR / "cell_02_input_validation_gate.csv", index=False)
print("Notebook 09 Cell 2 validation gate:")
display(notebook_09_cell_02_gate)
print("\nInput summary:")
display(source_summary)
failed = [name for name, passed in cell_02_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 09 Cell 2 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 2 completed successfully.")

In [ ]:
# Join the exact preceding bank quarter; never substitute an older row

if not notebook_09_cell_02_gate["passed"].all():
    raise AssertionError("Cell 3 blocked because Cell 2 did not pass.")

bank_source = predictor_source[["report_date", "cert", "idrssd"] + BANK_PREDICTORS_V1].copy()
bank_source = bank_source.rename(columns={"report_date": "source_report_date", "cert": "cert_source"})

if "insufficient_history_5q" in predictor_source.columns:
    history_source = predictor_source[["report_date", "idrssd", "insufficient_history_5q"]].copy()
    history_source = history_source.rename(columns={"report_date": "source_report_date"})
    history_source["insufficient_history_5q"] = pd.to_numeric(
        history_source["insufficient_history_5q"], errors="raise"
    ).astype("int8")
else:
    history_columns = ["report_date", "cert", "idrssd", "five_quarter_window_status"]
    history_source = pd.read_parquet(FULL_WORKING_SOURCE_PATH, columns=history_columns)
    history_source = normalise_quarter_end(history_source)
    history_source["idrssd"] = pd.to_numeric(history_source["idrssd"], errors="raise").astype("int64")
    history_source["cert"] = pd.to_numeric(history_source["cert"], errors="raise").astype("int64")
    history_source["insufficient_history_5q"] = (~history_source["five_quarter_window_status"].eq(
        "Eligible calendar window: five consecutive quarters"
    )).astype("int8")
    history_source = history_source.rename(columns={"report_date": "source_report_date", "cert": "cert_history_source"})
    history_source = history_source[["source_report_date", "idrssd", "cert_history_source", "insufficient_history_5q"]]

bank_source = bank_source.merge(
    history_source,
    on=["idrssd", "source_report_date"],
    how="left",
    validate="one_to_one",
)
if "cert_history_source" in bank_source.columns:
    if not bank_source["cert_source"].eq(bank_source["cert_history_source"]).all():
        raise AssertionError("Notebook 05 compact and full-working source certificates disagree.")
    bank_source = bank_source.drop(columns="cert_history_source")

reference_hash_before = int(pd.util.hash_pandas_object(
    target_reference[["_reference_row_id", "report_date", "cert_reference", "idrssd", "failed_1yr"]],
    index=False,
).sum())
target_reference["expected_bank_source_date"] = (
    target_reference["report_date"] - pd.offsets.QuarterEnd(1)
)
bank_exact_merge = target_reference.merge(
    bank_source,
    left_on=["idrssd", "expected_bank_source_date"],
    right_on=["idrssd", "source_report_date"],
    how="left",
    validate="one_to_one",
    indicator="bank_source_match",
)
no_exact_bank_report = bank_exact_merge.loc[
    bank_exact_merge["bank_source_match"].eq("left_only")
].copy()
exact_bank_panel = bank_exact_merge.loc[
    bank_exact_merge["bank_source_match"].eq("both")
].drop(columns="bank_source_match").copy()

reference_hash_after = int(pd.util.hash_pandas_object(
    bank_exact_merge[["_reference_row_id", "report_date", "cert_reference", "idrssd", "failed_1yr"]]
    .sort_values("_reference_row_id").reset_index(drop=True),
    index=False,
).sum())

bank_match_summary = pd.DataFrame({
    "measure": [
        "Restricted Notebook 07 reference observations", "Exact t-1 bank reports matched",
        "Reference observations with no exact t-1 bank report", "Positive rows lost",
        "Distinct failed banks lost", "Predictor-level missing cells retained among exact matches",
    ],
    "value": [
        len(target_reference), len(exact_bank_panel), len(no_exact_bank_report),
        int(no_exact_bank_report.failed_1yr.sum()),
        int(no_exact_bank_report.loc[no_exact_bank_report.failed_1yr.eq(1), "cert_reference"].nunique()),
        int(exact_bank_panel[BANK_PREDICTORS_V1].isna().sum().sum()),
    ],
})
cell_03_checks = {
    "Notebook 05 source keys are unique before the merge": not bank_source.duplicated(["idrssd", "source_report_date"]).any(),
    "The reference-to-source merge is one-to-one": len(bank_exact_merge) == len(target_reference),
    "Every retained source date is exactly the preceding quarter-end": exact_bank_panel.source_report_date.eq(exact_bank_panel.expected_bank_source_date).all(),
    "Every retained source date is before reference t": exact_bank_panel.source_report_date.lt(exact_bank_panel.report_date).all(),
    "No older source report was substituted": exact_bank_panel.source_report_date.eq(exact_bank_panel.report_date - pd.offsets.QuarterEnd(1)).all(),
    "Reference targets remain attached to their original rows": reference_hash_after == reference_hash_before,
    "The history indicator is complete and binary at its source quarter": exact_bank_panel.insufficient_history_5q.notna().all() and set(exact_bank_panel.insufficient_history_5q.unique()) <= {0, 1},
    "Predictor-level missingness is preserved": int(exact_bank_panel[BANK_PREDICTORS_V1].isna().sum().sum()) >= 0,
    "No retained bank predictor is infinite": not np.isinf(exact_bank_panel[BANK_PREDICTORS_V1].to_numpy(float)).any(),
}
notebook_09_cell_03_gate = pd.DataFrame({
    "check": list(cell_03_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_03_checks.values()],
    "passed": list(cell_03_checks.values()),
})
bank_match_summary.to_csv(NB09_OUTPUT_DIR / "cell_03_exact_bank_lag_match_summary.csv", index=False)
no_exact_bank_report[["report_date", "cert_reference", "idrssd", "failed_1yr", "expected_bank_source_date"]].to_csv(
    NB09_OUTPUT_DIR / "cell_03_no_exact_tminus1_bank_report.csv", index=False
)
notebook_09_cell_03_gate.to_csv(NB09_OUTPUT_DIR / "cell_03_validation_gate.csv", index=False)
print("Notebook 09 Cell 3 validation gate:")
display(notebook_09_cell_03_gate)
print("\nExact-quarter bank lag match summary:")
display(bank_match_summary)
failed = [name for name, passed in cell_03_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 09 Cell 3 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 3 completed successfully.")

In [ ]:
# Complete CERT scan and enforce the one-time adjudication stop

if not notebook_09_cell_03_gate["passed"].all():
    raise AssertionError("Cell 4 blocked because Cell 3 did not pass.")

CERT_KEY_COLUMNS = [
    "idrssd", "source_report_date", "report_date", "cert_source", "cert_reference"
]
ALLOW_DISPOSITION = "verified_charter_conversion_or_merger"
EXCLUDE_DISPOSITION = "crosswalk_or_identifier_error"
VALID_DISPOSITIONS = {ALLOW_DISPOSITION, EXCLUDE_DISPOSITION}
ADJUDICATION_VERSION = "v2_cert_adjudication_v1"

CERT_MISMATCH_PATH = NB09_OUTPUT_DIR / "v2_cert_mismatch_complete_scan.csv"
CERT_REGISTER_PATH = NB09_OUTPUT_DIR / "v2_cert_mismatch_adjudication_register.csv"
CERT_ALLOWLIST_PATH = NB09_OUTPUT_DIR / "v2_cert_mismatch_allowlist.csv"
CERT_EXCLUSION_PATH = NB09_OUTPUT_DIR / "v2_cert_mismatch_exclusion_list.csv"

cert_mismatch_table = exact_bank_panel.loc[
    exact_bank_panel.cert_source.ne(exact_bank_panel.cert_reference),
    CERT_KEY_COLUMNS + ["_reference_row_id", "failed_1yr"],
].sort_values(CERT_KEY_COLUMNS, kind="mergesort").reset_index(drop=True)
cert_mismatch_table.to_csv(CERT_MISMATCH_PATH, index=False)

register_columns = CERT_KEY_COLUMNS + [
    "disposition", "adjudication_note", "evidence_reference", "adjudication_version"
]
register_existed_at_start = CERT_REGISTER_PATH.exists()
if register_existed_at_start:
    saved_register = pd.read_csv(
        CERT_REGISTER_PATH,
        parse_dates=["source_report_date", "report_date"],
    )
    for column in ["idrssd", "cert_source", "cert_reference"]:
        saved_register[column] = pd.to_numeric(saved_register[column], errors="raise").astype("int64")
    for column in ["disposition", "adjudication_note", "evidence_reference", "adjudication_version"]:
        saved_register[column] = saved_register[column].fillna("").astype(str).str.strip()
    if saved_register.duplicated(CERT_KEY_COLUMNS).any():
        raise AssertionError("The CERT adjudication register contains duplicate keys.")
else:
    saved_register = pd.DataFrame(columns=register_columns)

current_keys = cert_mismatch_table[CERT_KEY_COLUMNS].drop_duplicates()
if current_keys.empty:
    current_adjudication = pd.DataFrame(columns=register_columns)
else:
    current_adjudication = current_keys.merge(
        saved_register[register_columns], on=CERT_KEY_COLUMNS, how="left", validate="one_to_one"
    )
for column in ["disposition", "adjudication_note", "evidence_reference", "adjudication_version"]:
    current_adjudication[column] = current_adjudication[column].fillna("").astype(str).str.strip()
current_adjudication.loc[current_adjudication.adjudication_version.eq(""), "adjudication_version"] = ADJUDICATION_VERSION

if current_keys.empty:
    stale_register = saved_register[register_columns].copy()
else:
    stale_register = saved_register.merge(
        current_keys, on=CERT_KEY_COLUMNS, how="left", indicator=True
    ).loc[lambda frame: frame._merge.eq("left_only"), register_columns]
updated_register = pd.concat([current_adjudication, stale_register], ignore_index=True)
updated_register = updated_register.sort_values(CERT_KEY_COLUMNS, kind="mergesort").reset_index(drop=True)
updated_register.to_csv(CERT_REGISTER_PATH, index=False)

unresolved_mask = (
    ~current_adjudication.disposition.isin(VALID_DISPOSITIONS)
    | current_adjudication.adjudication_note.eq("")
    | current_adjudication.evidence_reference.eq("")
)
unresolved_adjudications = current_adjudication.loc[unresolved_mask].copy()

if len(cert_mismatch_table) and (not register_existed_at_start or len(unresolved_adjudications)):
    print("Complete CERT mismatch scan:")
    display(cert_mismatch_table)
    print("\nAdjudication register requiring completion:")
    display(unresolved_adjudications)
    raise AssertionError(
        "Notebook 09 stopped once after the complete first-pass CERT scan, before writing "
        "the modelling panel. Complete disposition, adjudication_note and evidence_reference "
        f"in {CERT_REGISTER_PATH.name}. Allowed dispositions are "
        f"'{ALLOW_DISPOSITION}' and '{EXCLUDE_DISPOSITION}', then rerun all cells."
    )

cert_allowlist = current_adjudication.loc[
    current_adjudication.disposition.eq(ALLOW_DISPOSITION)
].copy()
cert_exclusion_list = current_adjudication.loc[
    current_adjudication.disposition.eq(EXCLUDE_DISPOSITION)
].copy()
cert_allowlist.to_csv(CERT_ALLOWLIST_PATH, index=False)
cert_exclusion_list.to_csv(CERT_EXCLUSION_PATH, index=False)

allow_keys = set(map(tuple, cert_allowlist[CERT_KEY_COLUMNS].itertuples(index=False, name=None)))
exclude_keys = set(map(tuple, cert_exclusion_list[CERT_KEY_COLUMNS].itertuples(index=False, name=None)))
mismatch_keys = set(map(tuple, current_keys[CERT_KEY_COLUMNS].itertuples(index=False, name=None)))
cell_04_checks = {
    "The full first-pass scan covered every exact bank match": len(exact_bank_panel) >= len(cert_mismatch_table),
    "Every current mismatch has exactly one allowed disposition": not unresolved_mask.any(),
    "The allowlist and exclusion list are disjoint": allow_keys.isdisjoint(exclude_keys),
    "The allowlist and exclusion list cover the complete current mismatch table": allow_keys | exclude_keys == mismatch_keys,
    "The CERT audit uses explicit source and reference certificate names": {"cert_source", "cert_reference"}.issubset(cert_mismatch_table.columns) and "cert" not in cert_mismatch_table.columns,
    "The adjudication allowlist is explicitly versioned": cert_allowlist.empty or cert_allowlist.adjudication_version.eq(ADJUDICATION_VERSION).all(),
}
notebook_09_cell_04_gate = pd.DataFrame({
    "check": list(cell_04_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_04_checks.values()],
    "passed": list(cell_04_checks.values()),
})
notebook_09_cell_04_gate.to_csv(NB09_OUTPUT_DIR / "cell_04_validation_gate.csv", index=False)
print("Notebook 09 Cell 4 validation gate:")
display(notebook_09_cell_04_gate)
print("\nCERT mismatch disposition counts:")
display(current_adjudication.disposition.value_counts(dropna=False).rename_axis("disposition").reset_index(name="rows"))
failed = [name for name, passed in cell_04_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 09 Cell 4 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 4 completed successfully.")

In [ ]:
# Apply CERT decisions, rename bank features, bind the calendar and merge macro once

if not notebook_09_cell_04_gate["passed"].all():
    raise AssertionError("Cell 5 blocked because Cell 4 did not pass.")

cert_decisions = current_adjudication[CERT_KEY_COLUMNS + ["disposition"]].copy()
adjudicated_bank_panel = exact_bank_panel.merge(
    cert_decisions, on=CERT_KEY_COLUMNS, how="left", validate="one_to_one"
)
adjudicated_bank_panel["disposition"] = adjudicated_bank_panel["disposition"].fillna("certificate_match")
crosswalk_identifier_exclusions = adjudicated_bank_panel.loc[
    adjudicated_bank_panel.disposition.eq(EXCLUDE_DISPOSITION)
].copy()
retained_bank_panel = adjudicated_bank_panel.loc[
    ~adjudicated_bank_panel.disposition.eq(EXCLUDE_DISPOSITION)
].copy()

rename_bank_features = {name: f"{name}_lag1" for name in BANK_PREDICTORS_V1}
rename_bank_features["insufficient_history_5q"] = HISTORY_FEATURE_LAG1
retained_bank_panel = retained_bank_panel.rename(columns=rename_bank_features)

surviving_reference_calendar = pd.DatetimeIndex(
    sorted(retained_bank_panel.report_date.unique())
)
expected_calendar = pd.period_range(
    surviving_reference_calendar.min().to_period("Q"),
    surviving_reference_calendar.max().to_period("Q"),
    freq="Q",
).to_timestamp("Q")

# Binding check: trim the full Notebook 08 panel first, then assert equality.
macro_trimmed = macro_full_source.loc[
    macro_full_source.report_date.isin(surviving_reference_calendar)
].copy()
macro_rows_outside_reference_calendar = len(macro_full_source) - len(macro_trimmed)
macro_mapping_trimmed = macro_lag_source.loc[
    macro_lag_source.report_date.isin(surviving_reference_calendar)
].copy()

reference_attachment_before = retained_bank_panel[
    ["_reference_row_id", "report_date", "cert_reference", "idrssd", "failed_1yr"]
].sort_values("_reference_row_id").reset_index(drop=True)
reference_attachment_hash_before = int(pd.util.hash_pandas_object(reference_attachment_before, index=False).sum())

panel_with_macro = retained_bank_panel.merge(
    macro_trimmed,
    on="report_date",
    how="left",
    validate="many_to_one",
    indicator="macro_match",
)
unmatched_macro_reference_rows = panel_with_macro.loc[
    panel_with_macro.macro_match.ne("both")
].copy()
panel_with_macro = panel_with_macro.drop(columns="macro_match")
reference_attachment_after = panel_with_macro[
    ["_reference_row_id", "report_date", "cert_reference", "idrssd", "failed_1yr"]
].sort_values("_reference_row_id").reset_index(drop=True)
reference_attachment_hash_after = int(pd.util.hash_pandas_object(reference_attachment_after, index=False).sum())

macro_binding_summary = pd.DataFrame({
    "measure": [
        "Surviving reference quarters", "First surviving reference quarter",
        "Last surviving reference quarter", "Macro rows before trimming",
        "Macro rows retained after trimming", "Macro rows outside reference calendar",
        "Unmatched surviving reference rows", "Macro lag applications",
    ],
    "value": [
        len(surviving_reference_calendar), surviving_reference_calendar.min(),
        surviving_reference_calendar.max(), len(macro_full_source), len(macro_trimmed),
        macro_rows_outside_reference_calendar, len(unmatched_macro_reference_rows), 1,
    ],
})
cell_05_checks = {
    "Verified charter conversions and mergers are retained unchanged": len(retained_bank_panel.loc[retained_bank_panel.disposition.eq(ALLOW_DISPOSITION)]) == len(cert_allowlist),
    "Crosswalk or identifier errors are excluded": len(crosswalk_identifier_exclusions) == len(cert_exclusion_list),
    "All bank modelling features end in lag1": all(name.endswith("_lag1") for name in BANK_FEATURES_LAG1 + [HISTORY_FEATURE_LAG1]),
    "No unsuffixed bank predictor survives into the feature set": set(BANK_PREDICTORS_V1 + ["insufficient_history_5q"]).isdisjoint(MODEL_FEATURES),
    "The surviving reference calendar begins in 2001Q2": surviving_reference_calendar.min() == REFERENCE_CALENDAR_START,
    "The surviving reference calendar has no interior gaps": surviving_reference_calendar.equals(expected_calendar),
    "The realised quarter count is derived from the surviving calendar": len(surviving_reference_calendar) == len(expected_calendar),
    "Macro was trimmed before the binding equality check": pd.DatetimeIndex(macro_trimmed.report_date).equals(surviving_reference_calendar),
    "Every surviving reference date has exactly one macro row": macro_trimmed.report_date.is_unique and set(macro_trimmed.report_date) == set(surviving_reference_calendar),
    "No surviving reference row is unmatched to macro": len(unmatched_macro_reference_rows) == 0,
    "Every macro source quarter is exactly t-1": macro_mapping_trimmed.source_quarter.eq(macro_mapping_trimmed.report_date - pd.offsets.QuarterEnd(1)).all(),
    "Macro was lagged once and not again": macro_mapping_trimmed.macro_lag_applications.eq(1).all(),
    "Targets remain on their original reference rows after the macro merge": reference_attachment_hash_after == reference_attachment_hash_before,
    "All merged macro values are complete and finite": panel_with_macro[MACRO_PREDICTORS].notna().all().all() and np.isfinite(panel_with_macro[MACRO_PREDICTORS].to_numpy(float)).all(),
    "No bank-quarter key is duplicated": not panel_with_macro.duplicated(["idrssd", "report_date"]).any(),
}
notebook_09_cell_05_gate = pd.DataFrame({
    "check": list(cell_05_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_05_checks.values()],
    "passed": list(cell_05_checks.values()),
})
macro_binding_summary.to_csv(NB09_OUTPUT_DIR / "cell_05_macro_calendar_binding_summary.csv", index=False)
notebook_09_cell_05_gate.to_csv(NB09_OUTPUT_DIR / "cell_05_validation_gate.csv", index=False)
print("Notebook 09 Cell 5 validation gate:")
display(notebook_09_cell_05_gate)
print("\nBinding calendar and macro merge summary:")
display(macro_binding_summary)
failed = [name for name, passed in cell_05_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 09 Cell 5 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 5 completed successfully.")

In [ ]:
# Audit positive-row predictor horizons and nominal filing availability

if not notebook_09_cell_05_gate["passed"].all():
    raise AssertionError("Cell 6 blocked because Cell 5 did not pass.")

positive_failure_lookup = target_audit_reference.loc[
    target_audit_reference.failed_1yr.eq(1),
    ["report_date", "cert_reference", "idrssd", "fail_date", "days_to_failure", "target_reason"],
].copy()
positive_availability_audit = panel_with_macro.loc[
    panel_with_macro.failed_1yr.eq(1),
    ["_reference_row_id", "report_date", "source_report_date", "expected_bank_source_date", "cert_reference", "idrssd", "failed_1yr"],
].merge(
    positive_failure_lookup,
    on=["report_date", "cert_reference", "idrssd"],
    how="left",
    validate="one_to_one",
)
positive_availability_audit["nominal_due_date_30"] = positive_availability_audit.source_report_date + pd.Timedelta(days=30)
positive_availability_audit["nominal_due_date_35"] = positive_availability_audit.source_report_date + pd.Timedelta(days=35)
positive_availability_audit["predictor_to_failure_days"] = (
    positive_availability_audit.fail_date - positive_availability_audit.source_report_date
).dt.days
positive_availability_audit["margin_after_30_day_due_date"] = (
    positive_availability_audit.fail_date - positive_availability_audit.nominal_due_date_30
).dt.days
positive_availability_audit["margin_after_35_day_due_date"] = (
    positive_availability_audit.fail_date - positive_availability_audit.nominal_due_date_35
).dt.days

distribution_columns = [
    "predictor_to_failure_days", "margin_after_30_day_due_date",
    "margin_after_35_day_due_date",
]
availability_summary_records = []
for scope, mask in [
    ("overall", pd.Series(True, index=positive_availability_audit.index)),
    ("final_test", positive_availability_audit.report_date.between(FINAL_TEST_START, FINAL_TEST_END)),
]:
    scoped = positive_availability_audit.loc[mask]
    for column in distribution_columns:
        values = scoped[column].dropna()
        availability_summary_records.append({
            "scope": scope, "distribution": column, "observations": len(values),
            "minimum_days": values.min(), "fifth_percentile_days": values.quantile(0.05),
            "median_days": values.median(), "maximum_days": values.max(),
        })
availability_margin_summary = pd.DataFrame(availability_summary_records)

timing_qualification = pd.DataFrame({
    "timing_audit": ["Bank filing availability", "Bank filing availability", "Macroeconomic publication timing", "Effective predictor-date horizon"],
    "rule": [
        "Source quarter-end + 30 days", "Source quarter-end + 35 days",
        "Use Notebook 08 series-specific nominal publication lags for t-1",
        "The target remains anchored at reference t, one quarter after the bank source date",
    ],
    "result": [
        bool(positive_availability_audit.margin_after_30_day_due_date.ge(0).all()),
        bool(positive_availability_audit.margin_after_35_day_due_date.ge(0).all()),
        bool(macro_timing_source.select_dtypes(include=["bool"]).all().all()) if len(macro_timing_source.select_dtypes(include=["bool"]).columns) else "See Notebook 08 audit",
        "Longer than Version 1 because Y(t,t+365] is measured from t while bank predictors come from t-1",
    ],
    "qualification": [
        "Nominal check only; actual historical filing timestamps and exceptional extensions are unavailable",
        "Nominal conservative check only; actual historical filing timestamps and exceptional extensions are unavailable",
        "Nominal check only; current-vintage macro data and no exact historical release timestamps",
        "Timing statement; no failure date or horizon enters the feature matrix",
    ],
})

cell_06_checks = {
    "Every retained positive row matched its Notebook 07 failure date": positive_availability_audit.fail_date.notna().all(),
    "Source-to-failure days reconcile to target days plus the t-1 to t gap": positive_availability_audit.predictor_to_failure_days.eq(positive_availability_audit.days_to_failure + (positive_availability_audit.report_date - positive_availability_audit.source_report_date).dt.days).all(),
    "No positive failure precedes the nominal 30-day due date": positive_availability_audit.margin_after_30_day_due_date.ge(0).all(),
    "No positive failure precedes the nominal 35-day due date": positive_availability_audit.margin_after_35_day_due_date.ge(0).all(),
    "Overall and final-test summaries contain all three distributions": set(availability_margin_summary.scope) == {"overall", "final_test"} and set(availability_margin_summary.distribution) == set(distribution_columns),
    "Failure dates and availability fields remain audit-only": set(["fail_date"] + distribution_columns).isdisjoint(MODEL_FEATURES),
}
notebook_09_cell_06_gate = pd.DataFrame({
    "check": list(cell_06_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_06_checks.values()],
    "passed": list(cell_06_checks.values()),
})
AVAILABILITY_AUDIT_PATH = NB09_OUTPUT_DIR / "v2_positive_predictor_availability_margin_audit.parquet"
positive_availability_audit.to_parquet(AVAILABILITY_AUDIT_PATH, index=False, compression="snappy")
availability_margin_summary.to_csv(NB09_OUTPUT_DIR / "v2_positive_predictor_availability_margin_summary.csv", index=False)
timing_qualification.to_csv(NB09_OUTPUT_DIR / "v2_bank_and_macro_nominal_timing_qualification.csv", index=False)
notebook_09_cell_06_gate.to_csv(NB09_OUTPUT_DIR / "cell_06_validation_gate.csv", index=False)
print("Notebook 09 Cell 6 validation gate:")
display(notebook_09_cell_06_gate)
print("\nPositive-row availability distributions:")
display(availability_margin_summary)
print("\nNominal timing qualification:")
with pd.option_context("display.max_colwidth", None):
    display(timing_qualification)
failed = [name for name, passed in cell_06_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 09 Cell 6 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 6 completed successfully.")

In [ ]:
# Recalculate exclusions, sample composition and predictor missingness

if not notebook_09_cell_06_gate["passed"].all():
    raise AssertionError("Cell 7 blocked because Cell 6 did not pass.")


def period_rows(frame, start, end):
    return frame.loc[frame.report_date.between(start, end)]


def summarise_frame(frame, label, period_type, start, end):
    scoped = period_rows(frame, start, end)
    positives = scoped.loc[scoped.failed_1yr.eq(1)]
    return {
        "period": label, "period_type": period_type, "start": start, "end": end,
        "observations": len(scoped), "positive_bank_quarters": int(scoped.failed_1yr.sum()),
        "distinct_failed_banks": int(positives.cert_reference.nunique()),
        "prevalence": float(scoped.failed_1yr.mean()) if len(scoped) else np.nan,
    }


exclusion_frames = {
    "no_exact_tminus1_bank_report": no_exact_bank_report,
    "crosswalk_or_identifier_error": crosswalk_identifier_exclusions,
}
exclusion_records = []
all_windows = [("overall", "overall", target_reference.report_date.min(), target_reference.report_date.max())] + PERIOD_WINDOWS
for reason, frame in exclusion_frames.items():
    for label, period_type, start, end in all_windows:
        row = summarise_frame(frame, label, period_type, start, end)
        row["exclusion_reason"] = reason
        exclusion_records.append(row)
exclusion_summary = pd.DataFrame(exclusion_records)[[
    "exclusion_reason", "period", "period_type", "start", "end", "observations",
    "positive_bank_quarters", "distinct_failed_banks", "prevalence",
]]

sample_summary = pd.DataFrame([
    summarise_frame(panel_with_macro, label, period_type, start, end)
    for label, period_type, start, end in all_windows
])
sample_summary["report_quarters"] = [
    period_rows(panel_with_macro, start, end).report_date.nunique()
    for _, _, start, end in all_windows
]

predictor_missingness = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "observations": len(panel_with_macro),
    "available_observations": [int(panel_with_macro[name].notna().sum()) for name in MODEL_FEATURES],
    "missing_observations": [int(panel_with_macro[name].isna().sum()) for name in MODEL_FEATURES],
    "missing_rate": [float(panel_with_macro[name].isna().mean()) for name in MODEL_FEATURES],
})
predictor_missingness["source_timing"] = np.where(
    predictor_missingness.feature.isin(MACRO_PREDICTORS), "macro t-1 from Notebook 08",
    "bank t-1 exact-quarter source",
)

cell_07_checks = {
    "Both exclusion reasons are reported for every chronological design window": set(exclusion_summary.exclusion_reason) == set(exclusion_frames) and set(exclusion_summary.period) == {row[0] for row in all_windows},
    "Training, purge, validation and test losses are explicitly reported": {"training", "purge", "validation", "test"}.issubset(exclusion_summary.period_type),
    "The final sample summary covers the realised calendar": sample_summary.loc[sample_summary.period.eq("overall"), "report_quarters"].iloc[0] == len(surviving_reference_calendar),
    "All 24 candidate features have a missingness record": list(predictor_missingness.feature) == MODEL_FEATURES,
    "Macro features contain no missing values": predictor_missingness.loc[predictor_missingness.feature.isin(MACRO_PREDICTORS), "missing_observations"].eq(0).all(),
    "Predictor-level bank missingness remains present rather than imputed": int(panel_with_macro[BANK_FEATURES_LAG1].isna().sum().sum()) == int(predictor_missingness.loc[predictor_missingness.feature.isin(BANK_FEATURES_LAG1), "missing_observations"].sum()),
    "No feature contains an infinite value": not np.isinf(panel_with_macro[MODEL_FEATURES].to_numpy(float)).any(),
}
notebook_09_cell_07_gate = pd.DataFrame({
    "check": list(cell_07_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_07_checks.values()],
    "passed": list(cell_07_checks.values()),
})
exclusion_summary.to_csv(NB09_OUTPUT_DIR / "v2_exclusion_summary.csv", index=False)
sample_summary.to_csv(NB09_OUTPUT_DIR / "v2_sample_summary.csv", index=False)
predictor_missingness.to_csv(NB09_OUTPUT_DIR / "v2_predictor_missingness_summary.csv", index=False)
notebook_09_cell_07_gate.to_csv(NB09_OUTPUT_DIR / "cell_07_validation_gate.csv", index=False)
print("Notebook 09 Cell 7 validation gate:")
display(notebook_09_cell_07_gate)
print("\nExclusions by chronological design period:")
display(exclusion_summary)
print("\nSurviving sample summary:")
display(sample_summary)
print("\nPredictor missingness (values retained unchanged):")
display(predictor_missingness)
failed = [name for name, passed in cell_07_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 09 Cell 7 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 7 completed successfully.")

In [ ]:
# Save the separate Version 2 panel and run the final leakage gate

if not notebook_09_cell_07_gate["passed"].all():
    raise AssertionError("Cell 8 blocked because Cell 7 did not pass.")

final_panel = panel_with_macro.rename(columns={"cert_reference": "cert"}).copy()
FINAL_PANEL_COLUMNS = ["report_date", "cert", "idrssd"] + MODEL_FEATURES + ["failed_1yr"]
final_panel = final_panel[FINAL_PANEL_COLUMNS].sort_values(
    ["report_date", "cert", "idrssd"], kind="mergesort"
).reset_index(drop=True)
final_panel["failed_1yr"] = final_panel.failed_1yr.astype("int8")
final_panel[HISTORY_FEATURE_LAG1] = final_panel[HISTORY_FEATURE_LAG1].astype("int8")
feature_matrix = final_panel[MODEL_FEATURES].copy()

lag_match_audit = adjudicated_bank_panel[[
    "_reference_row_id", "report_date", "expected_bank_source_date", "source_report_date",
    "idrssd", "cert_source", "cert_reference", "failed_1yr", "disposition",
]].copy()
lag_match_audit["exact_tminus1_match"] = lag_match_audit.source_report_date.eq(lag_match_audit.expected_bank_source_date)
lag_match_audit["certificate_mismatch"] = lag_match_audit.cert_source.ne(lag_match_audit.cert_reference)

FINAL_PANEL_PATH = NB09_OUTPUT_DIR / "v2_final_modelling_panel_2001Q2_2025Q2.parquet"
LAG_MATCH_AUDIT_PATH = NB09_OUTPUT_DIR / "v2_exact_bank_lag_match_audit.parquet"
final_panel.to_parquet(FINAL_PANEL_PATH, index=False, compression="snappy")
lag_match_audit.to_parquet(LAG_MATCH_AUDIT_PATH, index=False, compression="snappy")
final_recheck = pd.read_parquet(FINAL_PANEL_PATH)

feature_manifest = pd.DataFrame({
    "v1_notebook09_feature": LOCKED_V1_NOTEBOOK09_CANDIDATE_MANIFEST,
    "v2_notebook09_feature": MODEL_FEATURES,
    "feature_group": [
        "bank" if name in BANK_FEATURES_LAG1 else "macro" if name in MACRO_PREDICTORS else "history_indicator"
        for name in MODEL_FEATURES
    ],
    "timing": ["t-1"] * len(MODEL_FEATURES),
    "final_v2_selection": ["Deferred to Notebook 11"] * len(MODEL_FEATURES),
})

positives = final_panel.loc[final_panel.failed_1yr.eq(1)]
final_summary = pd.DataFrame({
    "measure": [
        "Analysis version", "Timing design", "Final observations", "Final columns",
        "Model features", "Bank features lagged exactly one quarter", "Macro features already lagged once",
        "First report date", "Last report date", "Realised reference quarters",
        "Positive bank-quarters", "Distinct failed banks", "Prevalence",
        "Rows lost without exact t-1 bank report", "Rows excluded for crosswalk or identifier errors",
        "Macro rows outside realised reference calendar", "Missing predictor cells",
        "Infinite predictor cells", "Values imputed", "Primary panel",
        "Lag-match audit", "CERT mismatch audit", "CERT allowlist", "CERT exclusion list",
        "Availability-margin audit", "Final Version 2 selection boundary",
    ],
    "value": [
        ANALYSIS_VERSION, "B(t-1), M(t-1) -> Y(t, t+365 days]", len(final_panel), len(final_panel.columns),
        len(MODEL_FEATURES), len(BANK_FEATURES_LAG1) + 1, len(MACRO_PREDICTORS),
        final_panel.report_date.min(), final_panel.report_date.max(), final_panel.report_date.nunique(),
        int(final_panel.failed_1yr.sum()), int(positives.cert.nunique()), float(final_panel.failed_1yr.mean()),
        len(no_exact_bank_report), len(crosswalk_identifier_exclusions), macro_rows_outside_reference_calendar,
        int(feature_matrix.isna().sum().sum()), int(np.isinf(feature_matrix.to_numpy(float)).sum()), 0,
        FINAL_PANEL_PATH.name, LAG_MATCH_AUDIT_PATH.name, CERT_MISMATCH_PATH.name,
        CERT_ALLOWLIST_PATH.name, CERT_EXCLUSION_PATH.name, AVAILABILITY_AUDIT_PATH.name,
        "Final V2 selection/manifest lock is deferred to Notebook 11",
    ],
})

cell_08_checks = {
    "The primary panel contains the exact locked Version 1 candidate manifest rendered for V2": list(feature_matrix.columns) == EXPECTED_V2_CANDIDATE_MANIFEST,
    "Every bank modelling feature ends in lag1": all(name.endswith("_lag1") for name in BANK_FEATURES_LAG1 + [HISTORY_FEATURE_LAG1]),
    "No unsuffixed bank predictor survives into the feature matrix": set(BANK_PREDICTORS_V1 + ["insufficient_history_5q"]).isdisjoint(feature_matrix.columns),
    "Identifiers, dates, certificates and target-derived fields are absent from the feature matrix": FEATURE_FORBIDDEN_FIELDS.isdisjoint(feature_matrix.columns),
    "The target is absent from the feature matrix": "failed_1yr" not in feature_matrix.columns,
    "Failure dates and horizons are absent from the primary panel": {"fail_date", "failure_date", "days_to_failure", "target_reason", "source_report_date", "expected_bank_source_date", "cert_source", "cert_reference"}.isdisjoint(final_panel.columns),
    "The final panel has unique IDRSSD-quarter keys": not final_panel.duplicated(["idrssd", "report_date"]).any(),
    "The final panel has unique CERT-quarter keys": not final_panel.duplicated(["cert", "report_date"]).any(),
    "The final panel begins in 2001Q2": final_panel.report_date.min() == REFERENCE_CALENDAR_START,
    "The final panel ends at the validated reference end": final_panel.report_date.max() == surviving_reference_calendar.max(),
    "The final calendar has no interior gaps": pd.DatetimeIndex(sorted(final_panel.report_date.unique())).equals(expected_calendar),
    "The realised quarter count is derived rather than hard-coded": final_panel.report_date.nunique() == len(expected_calendar),
    "All macro values remain attached once and are complete": final_panel[MACRO_PREDICTORS].notna().all().all(),
    "No feature contains an infinite value": not np.isinf(feature_matrix.to_numpy(float)).any(),
    "Predictor missingness was preserved without imputation": int(feature_matrix.isna().sum().sum()) == int(predictor_missingness.missing_observations.sum()),
    "The primary Parquet output exists": FINAL_PANEL_PATH.exists() and FINAL_PANEL_PATH.stat().st_size > 0,
    "The lag-match audit exists": LAG_MATCH_AUDIT_PATH.exists() and LAG_MATCH_AUDIT_PATH.stat().st_size > 0,
    "The primary Parquet output re-reads exactly": final_recheck.equals(final_panel),
    "All required audit outputs exist": all(path.exists() for path in [CERT_MISMATCH_PATH, CERT_ALLOWLIST_PATH, CERT_EXCLUSION_PATH, AVAILABILITY_AUDIT_PATH, NB09_OUTPUT_DIR / "v2_exclusion_summary.csv", NB09_OUTPUT_DIR / "v2_sample_summary.csv"]),
    "All previous Notebook 09 gates passed": all(gate.passed.all() for gate in [notebook_09_cell_01_gate, notebook_09_cell_02_gate, notebook_09_cell_03_gate, notebook_09_cell_04_gate, notebook_09_cell_05_gate, notebook_09_cell_06_gate, notebook_09_cell_07_gate]),
}
notebook_09_cell_08_gate = pd.DataFrame({
    "check": list(cell_08_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_08_checks.values()],
    "passed": list(cell_08_checks.values()),
})
feature_manifest.to_csv(NB09_OUTPUT_DIR / "v2_notebook09_feature_manifest.csv", index=False)
final_summary.to_csv(NB09_OUTPUT_DIR / "notebook_09_final_handoff_summary.csv", index=False)
notebook_09_cell_08_gate.to_csv(NB09_OUTPUT_DIR / "cell_08_final_validation_gate.csv", index=False)

print("Notebook 09 Cell 8 validation gate:")
display(notebook_09_cell_08_gate)
print("\nNotebook 09 final handoff summary:")
with pd.option_context("display.max_colwidth", None):
    display(final_summary)
print("\nVersion 2 modelling-panel preview:")
display(final_panel.head(10))
failed = [name for name, passed in cell_08_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 09 Cell 8 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 8 completed successfully.")
print("Notebook 09 is PASS / LOCKED.")
print("\nPrimary Version 2 modelling handoff:")
print(FINAL_PANEL_PATH)
print("\nExact-bank-lag audit:")
print(LAG_MATCH_AUDIT_PATH)
print("\nNotebook 10 can now define the Version 2 chronological design; final feature selection remains deferred to Notebook 11.")

In [ ]:
# Inspect certificate mismatch records

from pathlib import Path
import pandas as pd
from IPython.display import display

register_path = (
    NB09_OUTPUT_DIR
    / "v2_cert_mismatch_adjudication_register.csv"
)

cert_register = pd.read_csv(register_path)

print("CERT mismatches requiring adjudication:", len(cert_register))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(cert_register)

In [ ]:
# Verify historical certificate transitions

import numpy as np
import pandas as pd
from IPython.display import display

register_path = (
    NB09_OUTPUT_DIR
    / "v2_cert_mismatch_adjudication_register.csv"
)

register = pd.read_csv(register_path)

register["source_report_date"] = pd.to_datetime(
    register["source_report_date"]
).dt.normalize()

register["report_date"] = pd.to_datetime(
    register["report_date"]
).dt.normalize()

for column in ["idrssd", "cert_source", "cert_reference"]:
    register[column] = pd.to_numeric(
        register[column],
        errors="raise",
    ).astype("int64")


# Build the authoritative longitudinal CERT sequence from Notebook 05.
cert_history = predictor_source[
    ["idrssd", "report_date", "cert"]
].copy()

cert_history["report_date"] = pd.to_datetime(
    cert_history["report_date"]
).dt.normalize()

for column in ["idrssd", "cert"]:
    cert_history[column] = pd.to_numeric(
        cert_history[column],
        errors="raise",
    ).astype("int64")

if cert_history.duplicated(
    ["idrssd", "report_date"]
).any():
    raise AssertionError(
        "Notebook 05 contains duplicate IDRSSD-quarter keys."
    )

cert_lookup = cert_history.set_index(
    ["idrssd", "report_date"]
)["cert"]


def get_cert(idrssd, report_date):
    key = (int(idrssd), pd.Timestamp(report_date))

    if key not in cert_lookup.index:
        return np.nan

    return int(cert_lookup.loc[key])


verification_records = []

for row in register.itertuples(index=False):
    previous_date = (
        row.source_report_date
        - pd.offsets.QuarterEnd(1)
    )

    following_date = (
        row.report_date
        + pd.offsets.QuarterEnd(1)
    )

    previous_cert = get_cert(
        row.idrssd,
        previous_date,
    )

    actual_source_cert = get_cert(
        row.idrssd,
        row.source_report_date,
    )

    actual_reference_cert = get_cert(
        row.idrssd,
        row.report_date,
    )

    following_cert = get_cert(
        row.idrssd,
        following_date,
    )

    adjacent_quarters = (
        row.report_date
        == row.source_report_date
        + pd.offsets.QuarterEnd(1)
    )

    source_matches = (
        actual_source_cert
        == row.cert_source
    )

    reference_matches = (
        actual_reference_cert
        == row.cert_reference
    )

    previous_supports_source = (
        pd.isna(previous_cert)
        or previous_cert == row.cert_source
    )

    following_supports_reference = (
        pd.isna(following_cert)
        or following_cert == row.cert_reference
    )

    clean_transition = all([
        adjacent_quarters,
        source_matches,
        reference_matches,
        previous_supports_source,
        following_supports_reference,
    ])

    verification_records.append({
        "idrssd": row.idrssd,
        "source_report_date": row.source_report_date,
        "report_date": row.report_date,
        "previous_cert": previous_cert,
        "cert_source": row.cert_source,
        "actual_source_cert": actual_source_cert,
        "cert_reference": row.cert_reference,
        "actual_reference_cert": actual_reference_cert,
        "following_cert": following_cert,
        "adjacent_quarters": adjacent_quarters,
        "source_matches": source_matches,
        "reference_matches": reference_matches,
        "previous_supports_source": previous_supports_source,
        "following_supports_reference": following_supports_reference,
        "clean_transition": clean_transition,
    })


cert_transition_verification = pd.DataFrame(
    verification_records
)

failed_verification = (
    cert_transition_verification.loc[
        ~cert_transition_verification[
            "clean_transition"
        ]
    ]
)

print(
    "Transitions checked:",
    len(cert_transition_verification),
)

print(
    "Clean transitions:",
    int(
        cert_transition_verification[
            "clean_transition"
        ].sum()
    ),
)

print(
    "Transitions requiring further review:",
    len(failed_verification),
)

display(cert_transition_verification)


# Do not adjudicate automatically if any transition fails.
if len(failed_verification):
    display(failed_verification)

    raise AssertionError(
        "At least one CERT transition failed the "
        "longitudinal verification. Do not update "
        "the adjudication register yet."
    )


register["disposition"] = (
    "verified_charter_conversion_or_merger"
)

register["adjudication_note"] = (
    "Verified clean adjacent-quarter CERT transition "
    "for an unchanged IDRSSD. Notebook 05 confirms the "
    "source CERT at t-1 and reference CERT at t, with "
    "supporting surrounding-quarter continuity."
)

register["evidence_reference"] = (
    "Notebook 05 longitudinal IDRSSD-CERT sequence "
    "(t-2 through t+1); Federal Reserve RSSD9001 "
    "definition: ID_RSSD remains constant through "
    "charter or establishment changes."
)

register["adjudication_version"] = (
    "v2_cert_adjudication_v1"
)

register.to_csv(
    register_path,
    index=False,
)

cert_transition_verification.to_csv(
    NB09_OUTPUT_DIR
    / "v2_cert_transition_verification_audit.csv",
    index=False,
)

print(
    "\nAll 41 transitions passed. "
    "The adjudication register has been updated."
)

display(
    register[
        [
            "idrssd",
            "source_report_date",
            "report_date",
            "cert_source",
            "cert_reference",
            "disposition",
        ]
    ]
)